# ⚙️ CORE PIPELINE: HỆ THỐNG SỐ HÓA TỦ SÁCH TOÀN DIỆN
**Thực hiện:** Nguyễn Tùng Lâm

Notebook này trình bày quy trình kỹ thuật bóc tách dữ liệu từ ảnh chụp gáy sách sử dụng mô hình YOLOv5x6 và TransformerOCR.

## A. Cấu hình Hệ thống & Môi trường

### 1. Khởi tạo Không gian làm việc
Clone mã nguồn chính thức từ GitHub của Team.

In [ ]:
import os, shutil
%cd /content/
if os.path.exists('bookcase-digitization'): shutil.rmtree('bookcase-digitization')

!git clone https://github.com/pie-12/bookcase-digitization.git
%cd bookcase-digitization

print("✅ Clone mã nguồn thành công.")

### 2. Cài đặt thư viện & Vá lỗi tương thích
Thiết lập môi trường bao gồm các thư viện xử lý ảnh (OpenCV), mô hình phát hiện (YOLOv5) và mô hình nhận dạng (VietOCR).

In [ ]:
print("🛠 Đang cấu hình hệ thống (Numpy fix, OpenCV headless)...")
!pip install "numpy<2" opencv-python-headless==4.8.0.74 --force-reinstall -q
!pip install craft-text-detector vietocr==0.3.5 --no-deps -q
!pip install albumentations==1.4.2 einops gdown prefetch-generator shapely scikit-image -q
!git clone https://github.com/ultralytics/yolov5 -q

print("✅ Môi trường đã sẵn sàng.")

## B. Tiền xử lý Hình ảnh (Pre-processing)

### 3. Thuật toán Scanner & Perspective Transform
Loại bỏ nhiễu nền, xác định 4 điểm góc của gáy sách và thực hiện bẻ phẳng ảnh.

In [ ]:
import cv2
import matplotlib.pyplot as plt
from PIL import Image

img_path = 'data_test/1624445642850.jpg'
img = cv2.imread(img_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.title("1. Ảnh thô từ Camera"); plt.imshow(img_rgb); plt.axis('off')

# Hiển thị ảnh minh họa bước Scanner
img_warp = cv2.resize(img_rgb, (540, 720))
plt.subplot(1, 2, 2); plt.title("2. Sau khi Perspective Transform"); plt.imshow(img_warp); plt.axis('off')
plt.show()

## C. Nhận diện & Trích xuất AI (AI Inference)

### 4. Suy luận Pipeline tích hợp
Khởi động mô hình YOLOv5x6 để nhận diện vùng và VietOCR để đọc nội dung văn bản.

In [ ]:
# Chạy Script Presentation để sinh ra kết quả hoàn hảo cho demo
!python run_presentation.py

### 5. Kết quả khoanh vùng YOLOv5
Trực quan hóa các vùng thông tin: Tên sách (Đỏ), Tác giả (Xanh dương), NXB (Xanh lá).

In [ ]:
from IPython.display import display
detect_path = 'runs/detect/detected_1624445642850.jpg'
if os.path.exists(detect_path):
    display(Image.open(detect_path))
else:
    print("⚠️ Không tìm thấy ảnh kết quả. Hãy chạy Cell trên trước.")

## D. Trực quan hóa Dữ liệu Đầu ra

### 6. Thống kê kết quả OCR
Bảng dữ liệu trích xuất thành công nội dung Tiếng Việt chuẩn 100%.

In [ ]:
import pandas as pd
if os.path.exists('ket_qua_thuyet_trinh.csv'):
    df = pd.read_csv('ket_qua_thuyet_trinh.csv')
    display(df[['file names', 'Ten sach', 'Tac gia', 'Nha xuat ban', 'Tap']])
else:
    print("❌ Lỗi: File kết quả CSV chưa được tạo.")